Based on https://arxiv.org/pdf/1611.08024.

In [40]:
import torch
import torch.nn as nn

In [41]:
vocab_size = 4
num_channels = 64
num_samples = 128
dropout = 0.5
kern_length = 64
f1 = 8
f2 = 16
d = 2
norm_rate = 0.25

B = 32
T = num_samples 
C = num_channels

In [42]:
# input: (B, 1, C, T)

# --- block 1
conv2d = nn.Conv2d(
    in_channels=1,
    out_channels=f1,
    kernel_size=(1, kern_length),
    padding="same",
    bias=False,
) # (B, f1, C, T)

batch_norm1 = nn.BatchNorm2d(f1) # (B, f1, C, T)

depthwise_conv2d = nn.Conv2d(
    in_channels=f1,
    out_channels=d * f1,
    kernel_size=(C, 1),
    groups=f1,
    bias=False,
) # (B, d * f1, 1, T)

batch_norm2 = nn.BatchNorm2d(d * f1) # (B, d * f1, C, T)

elu = nn.ELU() # (B, d * f1, C, T)

avg_pool1 = nn.AvgPool2d(
    kernel_size=(1, 4),
) # (B, d * f1, 1, T / 4)

dropout = nn.Dropout2d(dropout) # (B, d * f1, 1, T / 4)

separable_conv2d = nn.Sequential(
    nn.Conv2d(
        in_channels=f1 * d,
        out_channels=f1 * d,
        kernel_size=(1, 16),
        padding="same",
        groups=f1 * d,
        bias=False,
    ),
    nn.Conv2d(
        f1 * d,
        f2,
        kernel_size=1,
        bias=False
    )
)

batch_norm3 = nn.BatchNorm2d(f2) # (B, f2, C, T)

avg_pool2 = nn.AvgPool2d(
    kernel_size=(1, 8),
) # (B, d * f1, 1, T / 32)

vocab_projection = nn.Linear(
    in_features=f2 * int(num_samples / 32),
    out_features=vocab_size,
)

In [43]:
x = torch.rand(B, T, C)
print(x.shape)
x = x.unsqueeze(1)
print(x.shape)
x = x.permute(0, 1, 3, 2)
print(x.shape)

x = conv2d(x)
print(x.shape)
x = batch_norm1(x)
print(x.shape)
x = depthwise_conv2d(x)
print(x.shape)
x = batch_norm2(x)
print(x.shape)
x = elu(x)
print(x.shape)
x = avg_pool1(x)
print(x.shape)
x = dropout(x)
print(x.shape)
x = separable_conv2d(x)
print(x.shape)
x = batch_norm3(x)
print(x.shape)
x = elu(x)
print(x.shape)
x = avg_pool2(x)
print(x.shape)
x = dropout(x)
print(x.shape)
x = x.flatten(start_dim=1)
print(x.shape)
x = vocab_projection(x)
print(x.shape)

torch.Size([32, 128, 64])
torch.Size([32, 1, 128, 64])
torch.Size([32, 1, 64, 128])
torch.Size([32, 8, 64, 128])
torch.Size([32, 8, 64, 128])
torch.Size([32, 16, 1, 128])
torch.Size([32, 16, 1, 128])
torch.Size([32, 16, 1, 128])
torch.Size([32, 16, 1, 32])
torch.Size([32, 16, 1, 32])
torch.Size([32, 16, 1, 32])
torch.Size([32, 16, 1, 32])
torch.Size([32, 16, 1, 32])
torch.Size([32, 16, 1, 4])
torch.Size([32, 16, 1, 4])
torch.Size([32, 64])
torch.Size([32, 4])
